# 05 — Đưa dữ liệu từ Gold sang MySQL

### 1. Kiểm tra bảng đầu vào

Notebook 04 tạo fact, dự đoán Spark và ngưỡng đơn trễ trong `pipeline_run_info`.
Xem mẫu fact và dự đoán trước khi tổng hợp.

In [ ]:
import sys

sys.path.insert(0, "/home/jovyan/pipeline")
import os

import pyspark.sql.functions as F
from pyspark.sql.window import Window
from steps import DIM, DIMS, FACT, PRED, RUN_INFO, get_spark

spark = get_spark("Delivery - Serving")

In [ ]:
fact = spark.read.format("delta").load(FACT)
predictions = spark.read.format("delta").load(PRED)
fact.select("order_id", "region_id", "delivery_minutes").show(3, truncate=False)
predictions.select("order_id", "delivery_minutes", "pred_lr", "pred_gbt").show(3, truncate=False)

### 2. Ghi và đọc lại số dòng

`serve_to_mysql()` tạo các bảng MySQL và đọc lại để đối soát.
Chạy cell ghi bảng khi Prefect đã dừng. Để dashboard nhận mã lần chạy
và trạng thái hoàn tất, chạy flow Prefect.

In [ ]:
def _jdbc():
    return {
        "url": os.getenv("MYSQL_JDBC_URL",
                         "jdbc:mysql://mysql:3306/delivery_source?allowPublicKeyRetrieval=true&useSSL=false"),
        "driver": "com.mysql.cj.jdbc.Driver",
        "user": os.getenv("MYSQL_USER", "delivery"),
        "password": os.getenv("MYSQL_PASSWORD", "delivery"),
    }


def write_mysql(df, table, *, run_id=None):
    if run_id is not None:
        df = df.withColumn("pipeline_run_id", F.lit(run_id))
    expected = df.count()
    if not expected:
        raise ValueError(f"Bảng {table} rỗng")
    options = {**_jdbc(), "dbtable": table}
    df.coalesce(1).write.format("jdbc").options(**options).mode("overwrite").save()
    written = df.sparkSession.read.format("jdbc").options(**options).load()
    if run_id is not None:
        result = written.agg(F.count("*").alias("total"),
                             F.count(F.when(F.col("pipeline_run_id") == run_id, 1)).alias("matched")).first()
        if result.total != result.matched:
            raise ValueError(f"MySQL {table}: dữ liệu thuộc nhiều lần chạy")
        actual = result.total
    else:
        actual = written.count()
    if actual != expected:
        raise ValueError(f"MySQL {table}: đã đọc lại {actual}, cần {expected} dòng")
    return actual

In [ ]:
def serving_frames(fact, pred, thr):
    late = (F.col("delivery_minutes") > F.lit(thr)).cast("double")

    reg = fact.groupBy("courier_id", "region_id").count()
    pick = Window.partitionBy("courier_id").orderBy(F.desc("count"), F.asc("region_id"))
    main = (reg.withColumn("rn", F.row_number().over(pick)).filter("rn = 1")
            .select("courier_id", F.col("region_id").alias("main_region_id")))

    cp = (pred.groupBy("courier_id").agg(
            F.round(F.avg("pred_gbt"), 2).alias("avg_pred_eta_minutes"),
            F.round(F.avg(F.abs(F.col("pred_gbt") - F.col("delivery_minutes"))), 2).alias("mae_gbt_test")))
    agg_courier = (fact.groupBy("courier_id").agg(
            F.count("order_id").alias("total_orders"),
            F.round(F.avg("distance_km"), 2).alias("avg_distance_km"),
            F.round(F.avg("delivery_minutes"), 2).alias("avg_delivery_minutes"),
            F.round(F.expr("percentile_approx(delivery_minutes, 0.5)"), 2).alias("median_delivery_minutes"),
            F.round(F.avg(late), 4).alias("late_rate"),
            F.round(F.max("delivery_minutes"), 2).alias("max_delivery_minutes"))
        .join(main, "courier_id", "left").join(cp, "courier_id", "left")
        .withColumn("late_threshold_min", F.lit(float(thr))))

    agg_region_daily = fact.groupBy("region_id", "order_date").agg(
        F.count("order_id").alias("total_orders"),
        F.countDistinct("courier_id").alias("active_couriers"),
        F.round(F.avg("distance_km"), 2).alias("avg_distance_km"),
        F.round(F.avg("delivery_minutes"), 2).alias("avg_delivery_minutes"),
        F.round(F.expr("percentile_approx(delivery_minutes, 0.5)"), 2).alias("median_delivery_minutes"),
        F.round(F.avg(late), 4).alias("late_rate"))

    agg_hourly = (fact.withColumn("accept_hour", F.hour("accept_ts")).groupBy("accept_hour").agg(
        F.count("order_id").alias("total_orders"),
        F.round(F.avg("delivery_minutes"), 2).alias("avg_delivery_minutes"),
        F.round(F.expr("percentile_approx(delivery_minutes, 0.5)"), 2).alias("median_delivery_minutes"),
        F.round(F.avg(late), 4).alias("late_rate")))

    over24 = (F.col("delivery_minutes") > 1440).cast("double")
    agg_region_summary = fact.groupBy("region_id").agg(
        F.count("order_id").alias("total_orders"),
        F.countDistinct("courier_id").alias("couriers"),
        F.round(F.avg("delivery_minutes"), 2).alias("avg_delivery_minutes"),
        F.round(F.expr("percentile_approx(delivery_minutes, 0.5)"), 2).alias("median_delivery_minutes"),
        F.round(F.expr("percentile_approx(delivery_minutes, 0.9)"), 2).alias("p90_delivery_minutes"),
        F.round(F.avg(late), 4).alias("late_rate"),
        F.round(F.avg(over24), 4).alias("over_24h_rate"))

    agg_overview = fact.agg(
        F.count("order_id").alias("total_orders"),
        F.countDistinct("courier_id").alias("couriers"),
        F.countDistinct("region_id").alias("regions"),
        F.round(F.avg("delivery_minutes"), 2).alias("avg_delivery_minutes"),
        F.round(F.expr("percentile_approx(delivery_minutes, 0.5)"), 2).alias("median_delivery_minutes"),
        F.round(F.expr("percentile_approx(delivery_minutes, 0.9)"), 2).alias("p90_delivery_minutes"),
        F.round(F.avg(late), 4).alias("late_rate"),
        F.round(F.avg(over24), 4).alias("over_24h_rate"))

    agg_region_error = pred.groupBy("region_id").agg(
        F.count("*").alias("n_test"),
        F.round(F.avg(F.abs(F.col("pred_median") - F.col("delivery_minutes"))), 2).alias("mae_baseline"),
        F.round(F.avg(F.abs(F.col("pred_lr") - F.col("delivery_minutes"))), 2).alias("mae_lr"),
        F.round(F.avg(F.abs(F.col("pred_gbt") - F.col("delivery_minutes"))), 2).alias("mae_gbt"))

    sample = (pred.orderBy(F.rand(42)).limit(3000).select(
        F.col("order_id").cast("string").alias("order_id"), "courier_id", "region_id", "aoi_type",
        F.round("distance_km", 2).alias("distance_km"), F.round("accept_hour", 2).alias("accept_hour"),
        F.round("delivery_minutes", 1).alias("actual_minutes"),
        F.round("pred_lr", 1).alias("pred_lr"), F.round("pred_gbt", 1).alias("pred_gbt")))

    return {
        "agg_courier_performance": agg_courier, "agg_region_daily": agg_region_daily,
        "agg_hourly": agg_hourly, "agg_region_error": agg_region_error,
        "agg_region_summary": agg_region_summary, "agg_overview": agg_overview,
        "eta_prediction_sample": sample,
    }

In [ ]:
def serve_to_mysql(spark, *, run_id=None):
    fact = spark.read.format("delta").load(FACT)
    pred = spark.read.format("delta").load(PRED)
    thr = spark.read.format("delta").load(RUN_INFO).first()["late_threshold_min"]
    out = {name: write_mysql(df, name, run_id=run_id)
           for name, df in serving_frames(fact, pred, thr).items()}
    for name, path in DIMS.items():  # dimension cho Metabase / truy vấn SQL
        out[name] = write_mysql(spark.read.format("delta").load(path), name, run_id=run_id)
    out["dim_courier"] = write_mysql(spark.read.format("delta").load(DIM), "dim_courier", run_id=run_id)
    return out

In [1]:
result = serve_to_mysql(spark)
result

{'agg_courier_performance': 203,
 'agg_region_daily': 3098,
 'agg_hourly': 19,
 'agg_region_error': 17,
 'agg_region_summary': 17,
 'agg_overview': 1,
 'eta_prediction_sample': 3000,
 'dim_region': 17,
 'dim_aoi': 8152,
 'dim_date': 189,
 'dim_hour': 24,
 'dim_courier': 226}